# Anexo A · Descarga de NASA VNP46A3 (cuaderno aparte)

**Proyecto Centinela · Fase 1 · v2 · Integrantes:** Tomás Rendón Bustamante · Juan Sebastián Gamba Aguirre

Este cuaderno **no** es necesario para correr el modelo (`01_centinela_mlp.ipynb`). Sirve para completar el Anexo A: nombre exacto, fecha de producción y SHA-256 de los diez archivos de luces nocturnas. Necesita una cuenta de NASA Earthdata (`earthaccess.login()` pide las credenciales; no se escriben en el cuaderno ni se suben al repositorio).

La imagen mide **actividad / iluminación, no homicidios**.


In [19]:
# ═══ A1. Conexión a NASA Earthdata y primera búsqueda ═══════════════
# Instalo la librería oficial para consultar los archivos de NASA. El «!» ejecuta un comando del sistema en Colab.
!pip install -q earthaccess
import earthaccess                # earthaccess: librería para buscar y descargar datos de NASA Earthdata
earthaccess.login()               # inicia sesión con mi cuenta Earthdata (el usuario y la contraseña NO se escriben en el cuaderno)
# Busco el producto VNP46A3 (luces nocturnas mensuales) de enero de 2021 sobre un recuadro que cubre Medellín (lon_min, lat_min, lon_max, lat_max)
res = earthaccess.search_data(short_name="VNP46A3",
                              temporal=("2021-01-01", "2021-01-31"),
                              bounding_box=(-75.8, 6.1, -75.4, 6.4))   # Medellín
print(len(res), [r["umm"]["GranuleUR"] for r in res])   # cuántos archivos («gránulos») encontró y su identificador
files = earthaccess.download(res, "data/raw/vnp46a3")   # los descargo a data/raw/vnp46a3
# La fecha de producción son los últimos 13 dígitos del nombre del archivo (AAAADDDHHMMSS).
# Con ella calculo el retraso: fecha de producción − fin del mes de la imagen. El SHA-256 de cada archivo va al anexo B1.


2 ['LAADS:8899944113', 'LAADS:8916465851']


In [20]:
# ═══ A2. Fecha de producción y SHA-256 de cada archivo descargado ═══
import os, glob, re, hashlib
from datetime import datetime, timedelta

def sha256(f):
    # Huella digital del archivo, leído por trozos de 1 MB (los archivos pesan decenas de MB)
    h = hashlib.sha256()
    with open(f, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

for f in sorted(glob.glob("data/raw/vnp46a3/*.h5")):
    n = os.path.basename(f)
    # El nombre tiene esta forma: VNP46A3.AAAADDD.tesela.versión.fechaProducción.h5; con una expresión regular separo sus partes
    m = re.match(r"VNP46A3\.A(\d{4})(\d{3})\.(h\d+v\d+)\.(\d{3})\.(\d{4})(\d{3})(\d{6})\.h5", n)
    ini = datetime.strptime(m.group(1) + m.group(2), "%Y%j")           # primer día del mes del compuesto (año + día del año)
    fin = (ini.replace(day=28) + timedelta(days=4)).replace(day=1) - timedelta(days=1)   # último día de ese mes
    prod = datetime.strptime(m.group(5) + m.group(6), "%Y%j")         # fecha de producción (año + día del año, primeros 7 de los 13 dígitos)
    print(n, f"{os.path.getsize(f)/1e6:.0f} MB", "tesela", m.group(3),
          "| producción", prod.date(), "| días tras fin de mes:", (prod - fin).days)
    print("   SHA-256:", sha256(f))


VNP46A3.A2020336.h10v08.002.2025134001101.h5 54 MB tesela h10v08 | producción 2025-05-14 | días tras fin de mes: 1595
   SHA-256: 74e4682ac5e7c84fff5b91d2290a8afa7bd937762d9e93e14f6fb8f77891b937
VNP46A3.A2021001.h10v08.002.2025147180816.h5 43 MB tesela h10v08 | producción 2025-05-27 | días tras fin de mes: 1577
   SHA-256: 10f2868c00c8cf362c9668235300ae7ad1307600c8062d73a5b56439b2b71181


In [21]:
# ═══ A3. Nombres exactos de los archivos de cada mes ═══════════════
# Para cada mes de imagen del Anexo A busco qué archivos devuelve NASA y guardo el nombre de cada uno.
# Cada búsqueda trae 2 archivos (el del mes y uno vecino); el que corresponde es el que empieza con el prefijo esperado del Anexo A.
import calendar
for ym in ["2021-09","2021-10","2022-01","2022-04","2022-06","2022-08","2022-10","2023-03","2023-08"]:
    y, mth = map(int, ym.split("-"))
    res = earthaccess.search_data(short_name="VNP46A3",
          temporal=(f"{ym}-01", f"{ym}-{calendar.monthrange(y, mth)[1]}"),
          bounding_box=(-75.8, 6.1, -75.4, 6.4))
    print(ym, len(res), [l.rsplit("/", 1)[-1] for r in res for l in r.data_links()])


2021-09 2 ['VNP46A3.A2021213.h10v08.002.2025149104901.h5', 'VNP46A3.A2021244.h10v08.002.2025148155946.h5']
2021-10 2 ['VNP46A3.A2021244.h10v08.002.2025148155946.h5', 'VNP46A3.A2021274.h10v08.002.2025148162844.h5']
2022-01 2 ['VNP46A3.A2021335.h10v08.002.2025148175415.h5', 'VNP46A3.A2022001.h10v08.002.2025155014535.h5']
2022-04 2 ['VNP46A3.A2022060.h10v08.002.2025155024057.h5', 'VNP46A3.A2022091.h10v08.002.2025155031414.h5']
2022-06 2 ['VNP46A3.A2022121.h10v08.002.2025155034509.h5', 'VNP46A3.A2022152.h10v08.002.2025155185700.h5']
2022-08 2 ['VNP46A3.A2022182.h10v08.002.2025155191348.h5', 'VNP46A3.A2022213.h10v08.002.2025155220025.h5']
2022-10 2 ['VNP46A3.A2022244.h10v08.002.2025155193117.h5', 'VNP46A3.A2022274.h10v08.002.2025155195249.h5']
2023-03 2 ['VNP46A3.A2023032.h10v08.002.2025160185049.h5', 'VNP46A3.A2023060.h10v08.002.2025160193310.h5']
2023-08 2 ['VNP46A3.A2023182.h10v08.002.2025161113106.h5', 'VNP46A3.A2023213.h10v08.002.2025160224217.h5']


In [ ]:
# ═══ A4. (ejecutada el 2026-11-08) Descargar los 10 archivos del Anexo A y calcular sus SHA-256 ═══
# Esta celda baja solo los diez archivos de la tabla del Anexo A (los que empiezan con el prefijo esperado)
# y luego vuelvo a correr la celda A2 para imprimir su fecha de producción y su SHA-256.
import pandas as pd
meses_img = ["2021-01","2021-09","2021-10","2022-01","2022-04","2022-06","2022-08","2022-10","2023-03","2023-08"]
a_bajar = []
for ym in meses_img:
    y, mth = map(int, ym.split("-"))
    prefijo = f"VNP46A3.A{y}{pd.Timestamp(ym + '-01').dayofyear:03d}.h10v08.002."      # prefijo esperado del archivo de ese mes
    res = earthaccess.search_data(short_name="VNP46A3",
          temporal=(f"{ym}-01", f"{ym}-{calendar.monthrange(y, mth)[1]}"),
          bounding_box=(-75.8, 6.1, -75.4, 6.4))
    for r in res:
        if any(l.rsplit("/", 1)[-1].startswith(prefijo) for l in r.data_links()):   # me quedo con el que coincide con el prefijo
            a_bajar.append(r)
print("Archivos a descargar:", len(a_bajar), "(deben ser 10)")
files = earthaccess.download(a_bajar, "data/raw/vnp46a3")
